# 05 - Friction Measures Van (v2, generated 2026-10-13)

Fabric folder: Dev_CGP > Notebooks > 03 - EADM > Collaboration. Run order: 01 - Policy Renewals 30d, 02 - Friction Frame, then this notebook (03 Car first,
which builds the shared call table stg.fe3m_calls_x).

The Van measures of the friction extract (spec v3.3) on the frame's episodes: one row per Van episode in
`stg.fe3m_van` with all 100 Van fields. Same rules as 03 - Friction Measures Car:
flags 1 / 0 on their own episode type, blank elsewhere; blank before the reliable-from date or when an eligible
episode meets neither rule; branches blank while open on DataDate; a long call = queue + ring time >=
`WaitThresholdSeconds` or no agent answered. Writes `stg.fe3m_van*` only.

Van (2026-10-13): built F1, F5, R2, R3 (as Home), C3, C4 and the branches VA5.B1, VA6.B1, VR2.B1,
VR4.B2 (base: Outcome_ThisRenewal = Renewed, spec), VM3.B1. Blank: F2, F3, F4, R4, C5 (spec: no source); R1, R5, R6
(no Van events); C1 (Van changes not captured, agreed). Not discriminating with this data: F5 and VA5.B1 (only the
chase step VA5.B1.4 exists, from May 2026); VR2.B1 (no VR2.B1 events). VM3.B1 is blank in practice: the frame has no
Van change episodes.
Branches are entered on their own spine (happy-path steps from the flow sheets), not on their friction or outcome steps.


Parameters. Reliable-from dates apply to EpisodeStart. RF_Calls: Genesys queues complete from Nov 2022 (F1, R2, R3, C3, C4). RF_F5, RF_Loan: VA5.B1.4 (the only VA5.B1 step in the stream) from mid May 2026. RF_FyDoc: VA6 from Aug 2023. RF_RnDoc, RF_ChangeDoc: document intake from Oct 2023 (as Car / Home). RF_Requote: VR2a logins from 2021 (no VR2.B1 events in the stream: the branch cannot be entered).

In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "ChillLH"
    }
}

In [1]:
Set WaitThresholdSeconds = 600 ;
Set PreSaleLookbackDays = 30 ;
Set RF_Calls = '2022-11-01' ;
Set RF_F5 = '2026-06-01' ;
Set RF_Loan = '2026-06-01' ;
Set RF_FyDoc = '2023-09-01' ;
Set RF_Requote = '2021-07-01' ;
Set RF_RnDoc = '2023-10-01' ;
Set RF_ChangeDoc = '2023-10-01' ;
Set RF_R2 = '2022-11-01' ;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 11, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

Van episodes from the frame, with the window end used by "before DecisionDate" rules and an open flag.

In [2]:
CREATE OR REPLACE TABLE stg.fe3m_van_ep AS
SELECT e.*,
       least(e.EpisodeEnd, COALESCE(e.DecisionDate, e.EpisodeEnd)) AS win_end,
       CASE WHEN e.EpisodeEnd > e.DataDate THEN 1 ELSE 0 END          AS is_open
FROM   stg.fe3_episode e
WHERE  e.Product = 'Van'
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Van events used by the measures (distinct: identical copies count once).

In [3]:
CREATE OR REPLACE TABLE stg.fe3m_van_events AS
SELECT DISTINCT EventTypeId, EventDate, TRIM(SourcePolicyReference) AS pc, SourceQuoteReference AS qr,
       CASE WHEN SCV_Customer_Key > 0 THEN SCV_Customer_Key END AS scv
FROM   ods.episodeeventsnumberedfinal
WHERE  ProductType = 'van' AND EventDate >= '2022-01-01' AND EventDate <= current_date()
AND   (EventTypeId IN ('VA6', 'VR2a', 'VR1a', 'VR1b', 'VR4', 'VM3', 'VA5')
       OR EventTypeId LIKE 'VA6.B1%' OR EventTypeId LIKE 'VR2.B1%' OR EventTypeId LIKE 'VR4.B2%' OR EventTypeId LIKE 'VM3.B1%' OR EventTypeId LIKE 'VA5.B1%' )
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 13, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

Call measures: per episode, the calls of a line inside the measure's window (calls from 03: stg.fe3m_calls_x).

In [4]:
CREATE OR REPLACE TABLE stg.fe3m_van_callagg AS
WITH w AS (
    SELECT e.EpisodeID, 'F1' AS m, c.*
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'sales' AND c.queue_product = 'Van'
           AND c.call_date BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, 'R2', c.*
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'renewal' AND c.queue_product = 'Van'
           AND c.call_date BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    UNION ALL
    SELECT e.EpisodeID, 'C1', c.*
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'change' AND c.queue_product = 'Van'
           AND c.call_date BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Change'
    UNION ALL
    SELECT e.EpisodeID, 'C3', c.*
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'document'
           AND c.call_date BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Document request'
),
w2 AS (
    SELECT w.*, MIN(CASE WHEN is_long = 1 THEN call_ts END) OVER (PARTITION BY EpisodeID, m) AS first_long
    FROM   w
)
SELECT EpisodeID, m, COUNT(*) AS n, SUM(connected) AS n_conn, SUM(1 - connected) AS n_aband,
       CAST(ROUND(MAX(wait_s)) AS INT) AS max_wait, SUM(is_long) AS n_long, MAX(first_long) AS first_long,
       MAX(CASE WHEN connected = 1 AND call_ts >= first_long THEN 1 ELSE 0 END) AS resolved
FROM   w2 GROUP BY EpisodeID, m
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

R3: first online session (login) per renewal episode, then whether a renewal-line call follows (date level).

In [5]:
CREATE OR REPLACE TABLE stg.fe3m_van_r3 AS
WITH l AS (
    SELECT e.EpisodeID, MIN(v.EventDate) AS first_login
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_van_events v
           ON v.scv = e.scv_id AND v.EventTypeId = 'VR2a' AND v.EventDate BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    GROUP  BY e.EpisodeID
)
SELECT l.EpisodeID, l.first_login,
       MAX(CASE WHEN c.conversationId IS NOT NULL THEN 1 ELSE 0 END) AS called_after
FROM   l JOIN stg.fe3m_van_ep e ON e.EpisodeID = l.EpisodeID
LEFT JOIN stg.fe3m_calls_x c
       ON c.scv_id = e.scv_id AND c.line = 'renewal' AND c.queue_product = 'Van'
       AND c.call_date >= l.first_login AND c.call_date <= e.win_end
GROUP  BY l.EpisodeID, l.first_login
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Renewal episodes under each code their events can carry: the policy and, if different, its renewed code.

In [6]:
CREATE OR REPLACE TABLE stg.fe3m_van_rcodes AS
SELECT EpisodeID, PolicyRef AS code, DueDate, EpisodeEnd, win_end FROM stg.fe3m_van_ep WHERE EpisodeType = 'Renewal'
UNION
SELECT EpisodeID, new_policy_code, DueDate, EpisodeEnd, win_end FROM stg.fe3m_van_ep
WHERE  EpisodeType = 'Renewal' AND new_policy_code IS NOT NULL AND new_policy_code <> PolicyRef
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Event aggregates per episode: quote-keyed first-year events by customer in the 30 days before the sale; policy- keyed events by the policy inside the episode (renewal events also under the renewed code); logins by customer.

**Notes**

- invitations, held letters and payment steps carry derived dates (due - 60 / due date): a wider band

In [7]:
CREATE OR REPLACE TABLE stg.fe3m_van_evagg AS
WITH x AS (
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_van_events v
           ON  v.scv = e.scv_id AND v.EventTypeId IN ('__none__', '__none__', '__none__', '__none__')
           AND v.EventDate BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_van_events v
           ON  v.pc = e.PolicyRef
           AND (v.EventTypeId IN ('VA6', 'VA5') OR v.EventTypeId LIKE 'VA6.B1%' OR v.EventTypeId LIKE 'VA5.B1%')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_van_events v
           ON  v.scv = e.scv_id AND v.EventTypeId IN ('VR2a', '__none__')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    UNION ALL
    SELECT r.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_van_rcodes r JOIN stg.fe3m_van_events v
           ON  v.pc = r.code
           AND (v.EventTypeId IN ('VR1a', 'VR1b', 'VR4') OR v.EventTypeId LIKE 'VR2.B1%' OR v.EventTypeId LIKE 'VR4.B2%' )
           AND v.EventDate BETWEEN date_sub(r.DueDate, 90) AND date_add(r.EpisodeEnd, 16)
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_van_ep e JOIN stg.fe3m_van_events v
           ON  v.pc = e.PolicyRef AND (v.EventTypeId = 'VM3' OR v.EventTypeId LIKE 'VM3.B1%')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Change'
)
SELECT x.EpisodeID,
       MAX(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END)                  AS q_online,
       MAX(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END)                     AS got_price,
       MAX(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END)               AS no_price_online,
       MAX(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END)                 AS bought_online,
       MAX(CASE WHEN EventTypeId IN ('__none__') THEN 1 ELSE 0 END)          AS bought_by_phone,
       MAX(CASE WHEN EventTypeId = 'VA6' THEN 1 ELSE 0 END)                AS fy_doc_reached,
       SUM(CASE WHEN EventTypeId IN ('VA6.B1', 'VA6.B1.1', 'VA6.B1.2', 'VA6.B1.3') THEN 1 ELSE 0 END) AS fy_doc_n,
       MIN(CASE WHEN EventTypeId = '__none__' THEN EventDate END)                     AS dispatch_first,
       SUM(CASE WHEN EventTypeId = 'VR2a' THEN 1 ELSE 0 END)                      AS logins_ok,
       SUM(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END)                  AS logins_failed,
       MIN(CASE WHEN EventTypeId = '__none__' THEN EventDate END)                 AS first_failed,
       MAX(CASE WHEN EventTypeId = 'VR2a' THEN EventDate END)                     AS last_ok,
       MAX(CASE WHEN EventTypeId = '__none__' AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS held,
       MIN(CASE WHEN EventTypeId = '__none__' AND EventDate <= e.DueDate THEN EventDate END) AS held_first,
       MAX(CASE WHEN EventTypeId IN ('VR1a', 'VR1b') AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS invited,
       MAX(CASE WHEN EventTypeId IN ('VR1a', 'VR1b') AND EventDate <= e.DueDate THEN EventDate END) AS invited_last,
       MAX(CASE WHEN EventTypeId = '__none__' AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS date_missed,
       SUM(CASE WHEN EventTypeId LIKE 'VR2.B1%' AND EventDate <= e.win_end THEN 1 ELSE 0 END) AS requote_n,
       SUM(CASE WHEN EventTypeId IN ('VR2.B1')
                AND EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd THEN 1 ELSE 0 END)   AS requote_win_n,
       COUNT(DISTINCT CASE WHEN EventTypeId = '__none__' AND EventDate <= e.win_end THEN COALESCE(qr, CAST(EventDate AS STRING)) END)
                                                                                            AS rq_with_price,
       SUM(CASE WHEN EventTypeId = '__none__' AND EventDate <= e.win_end THEN 1 ELSE 0 END) AS rq_no_price,
       MIN(CASE WHEN EventTypeId = '__none__' AND EventDate <= e.win_end THEN EventDate END) AS first_no_price,
       MAX(CASE WHEN EventTypeId = '__none__' AND EventDate <= e.win_end THEN EventDate END)    AS last_price,
       MAX(CASE WHEN EventTypeId = 'VR4' THEN 1 ELSE 0 END)                          AS paid,
       SUM(CASE WHEN EventTypeId = '__none__' THEN 1 ELSE 0 END)                   AS monthly_n,
       SUM(CASE WHEN EventTypeId IN ('VR4.B2', 'VR4.B2.1', 'VR4.B2.2', 'VR4.B2.3')
                AND EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd THEN 1 ELSE 0 END)   AS rn_doc_n,
       MAX(CASE WHEN EventTypeId = 'VA5' THEN 1 ELSE 0 END) AS loan_reached,
       SUM(CASE WHEN EventTypeId IN ('VA5.B1', 'VA5.B1.1') THEN 1 ELSE 0 END) AS loan_n,
       SUM(CASE WHEN EventTypeId LIKE 'VA5.B1%' AND EventDate >= e.SaleDate THEN 1 ELSE 0 END)
                                                                                            AS loan_after_sale_n,
       SUM(CASE WHEN EventTypeId = 'VA5.B1.4' AND EventDate >= e.SaleDate THEN 1 ELSE 0 END)
                                                                                            AS loan_chase_n,
       MAX(CASE WHEN EventTypeId = 'VM3' THEN 1 ELSE 0 END)                AS change_reached,
       SUM(CASE WHEN EventTypeId IN ('VM3.B1', 'VM3.B1.1', 'VM3.B1.2', 'VM3.B1.3') THEN 1 ELSE 0 END) AS change_doc_n
FROM   x JOIN stg.fe3m_van_ep e USING (EpisodeID)
GROUP  BY x.EpisodeID
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

F4: chases after the first dispatch; resolved if the documents are dispatched again after the first chase.

In [8]:
CREATE OR REPLACE TABLE stg.fe3m_van_ev2 AS
WITH k AS (
    SELECT a.EpisodeID, e.PolicyRef, e.EpisodeEnd, a.dispatch_first
    FROM   stg.fe3m_van_evagg a JOIN stg.fe3m_van_ep e ON e.EpisodeID = a.EpisodeID
    WHERE  e.EpisodeType = 'First year' AND a.dispatch_first IS NOT NULL
),
ch AS (
    SELECT k.EpisodeID, COUNT(v.EventDate) AS chases, MIN(v.EventDate) AS first_chase
    FROM   k LEFT JOIN stg.fe3m_van_events v
           ON v.pc = k.PolicyRef AND v.EventTypeId = '__none__'
           AND v.EventDate >= k.dispatch_first AND v.EventDate <= k.EpisodeEnd
    GROUP  BY k.EpisodeID
)
SELECT ch.EpisodeID, ch.chases, ch.first_chase,
       MAX(CASE WHEN v.EventDate > ch.first_chase AND v.EventDate <= k.EpisodeEnd THEN 1 ELSE 0 END) AS redispatched
FROM   ch JOIN k ON k.EpisodeID = ch.EpisodeID
LEFT JOIN stg.fe3m_van_events v ON v.pc = k.PolicyRef AND v.EventTypeId = '__none__'
GROUP  BY ch.EpisodeID, ch.chases, ch.first_chase
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

The Van measures: one row per Van episode, all 100 fields in the spec's order.

In [9]:
CREATE OR REPLACE TABLE stg.fe3m_van AS
SELECT e.EpisodeID, e.EpisodeType,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(f1.n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_FirstYear_F1_SalesCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(f1.n, 0) > 0) AND (COALESCE(f1.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Van_FirstYear_F1_SalesCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND (NOT (COALESCE(f1.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Van_FirstYear_F1_SalesCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT ((COALESCE(f1.n, 0) > 0) AND (COALESCE(f1.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN f1.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Van_FirstYear_F1_SalesCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n END
           AS Van_FirstYear_F1_SalesCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n_conn END
           AS Van_FirstYear_F1_SalesCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n_aband END
           AS Van_FirstYear_F1_SalesCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.max_wait END
           AS Van_FirstYear_F1_SalesCallWait_LongestWaitSeconds,
       CAST(NULL AS INT)
           AS Van_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible,
       CAST(NULL AS INT)
           AS Van_FirstYear_F2_StartedOnlineFinishedByPhone_Affected,
       CAST(NULL AS INT)
           AS Van_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison,
       CAST(NULL AS INT)
           AS Van_FirstYear_F3_NoPriceOnline_Eligible,
       CAST(NULL AS INT)
           AS Van_FirstYear_F3_NoPriceOnline_Affected,
       CAST(NULL AS INT)
           AS Van_FirstYear_F3_NoPriceOnline_Comparison,
       CAST(NULL AS INT)
           AS Van_FirstYear_F4_DocsChasedByCustomer_Eligible,
       CAST(NULL AS INT)
           AS Van_FirstYear_F4_DocsChasedByCustomer_Affected,
       CAST(NULL AS INT)
           AS Van_FirstYear_F4_DocsChasedByCustomer_Comparison,
       CAST(NULL AS STRING)
           AS Van_FirstYear_F4_DocsChasedByCustomer_Resolution,
       CAST(NULL AS INT)
           AS Van_FirstYear_F4_DocsChasedByCustomer_EventCount,
       CAST(NULL AS INT)
           AS Van_FirstYear_F4_DocsChasedByCustomer_DaysFromDispatchToFirstChase,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F5}) OR ((COALESCE(ev.loan_after_sale_n, 0) > 0) AND NOT (COALESCE(ev.loan_chase_n, 0) > 0) AND NOT (NOT (COALESCE(ev.loan_chase_n, 0) > 0)))) THEN NULL WHEN COALESCE(ev.loan_after_sale_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_FirstYear_F5_LoanSetupChase_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F5}) OR ((COALESCE(ev.loan_after_sale_n, 0) > 0) AND NOT (COALESCE(ev.loan_chase_n, 0) > 0) AND NOT (NOT (COALESCE(ev.loan_chase_n, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.loan_after_sale_n, 0) > 0) AND (COALESCE(ev.loan_chase_n, 0) > 0) THEN 1 ELSE 0 END
           AS Van_FirstYear_F5_LoanSetupChase_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F5}) OR ((COALESCE(ev.loan_after_sale_n, 0) > 0) AND NOT (COALESCE(ev.loan_chase_n, 0) > 0) AND NOT (NOT (COALESCE(ev.loan_chase_n, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.loan_after_sale_n, 0) > 0) AND NOT (COALESCE(ev.loan_chase_n, 0) > 0) AND (NOT (COALESCE(ev.loan_chase_n, 0) > 0)) THEN 1 ELSE 0 END
           AS Van_FirstYear_F5_LoanSetupChase_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F5}) OR ((COALESCE(ev.loan_after_sale_n, 0) > 0) AND NOT (COALESCE(ev.loan_chase_n, 0) > 0) AND NOT (NOT (COALESCE(ev.loan_chase_n, 0) > 0)))) OR NOT ((COALESCE(ev.loan_after_sale_n, 0) > 0) AND (COALESCE(ev.loan_chase_n, 0) > 0)) THEN NULL ELSE 'Unknown' END
           AS Van_FirstYear_F5_LoanSetupChase_Resolution,
       CAST(NULL AS INT)
           AS Van_Renewal_R1_FailedLogin_Eligible,
       CAST(NULL AS INT)
           AS Van_Renewal_R1_FailedLogin_Affected,
       CAST(NULL AS INT)
           AS Van_Renewal_R1_FailedLogin_Comparison,
       CAST(NULL AS STRING)
           AS Van_Renewal_R1_FailedLogin_Resolution,
       CAST(NULL AS INT)
           AS Van_Renewal_R1_FailedLogin_FailedLogins,
       CAST(NULL AS INT)
           AS Van_Renewal_R1_FailedLogin_SuccessfulLogins,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(r2.n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Renewal_R2_RenewalCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(r2.n, 0) > 0) AND (COALESCE(r2.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Van_Renewal_R2_RenewalCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND (NOT (COALESCE(r2.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Van_Renewal_R2_RenewalCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT ((COALESCE(r2.n, 0) > 0) AND (COALESCE(r2.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN r2.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Van_Renewal_R2_RenewalCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n END
           AS Van_Renewal_R2_RenewalCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n_conn END
           AS Van_Renewal_R2_RenewalCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n_aband END
           AS Van_Renewal_R2_RenewalCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.max_wait END
           AS Van_Renewal_R2_RenewalCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN r3.first_login IS NOT NULL THEN 1 ELSE 0 END
           AS Van_Renewal_R3_OnlineThenCalled_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN (r3.first_login IS NOT NULL) AND (COALESCE(r3.called_after, 0) = 1) THEN 1 ELSE 0 END
           AS Van_Renewal_R3_OnlineThenCalled_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls}) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN (r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND (NOT (COALESCE(r3.called_after, 0) = 1)) THEN 1 ELSE 0 END
           AS Van_Renewal_R3_OnlineThenCalled_Comparison,
       CAST(NULL AS INT)
           AS Van_Renewal_R4_RequoteNoPrice_Eligible,
       CAST(NULL AS INT)
           AS Van_Renewal_R4_RequoteNoPrice_Affected,
       CAST(NULL AS INT)
           AS Van_Renewal_R4_RequoteNoPrice_Comparison,
       CAST(NULL AS STRING)
           AS Van_Renewal_R4_RequoteNoPrice_Resolution,
       CAST(NULL AS INT)
           AS Van_Renewal_R4_RequoteNoPrice_RequotesWithPrice,
       CAST(NULL AS INT)
           AS Van_Renewal_R4_RequoteNoPrice_RequotesWithoutPrice,
       CAST(NULL AS INT)
           AS Van_Renewal_R4_RequoteNoPrice_RequotesAbandoned,
       CAST(NULL AS INT)
           AS Van_Renewal_R5_RenewalHeld_Eligible,
       CAST(NULL AS INT)
           AS Van_Renewal_R5_RenewalHeld_Affected,
       CAST(NULL AS INT)
           AS Van_Renewal_R5_RenewalHeld_Comparison,
       CAST(NULL AS STRING)
           AS Van_Renewal_R5_RenewalHeld_Resolution,
       CAST(NULL AS INT)
           AS Van_Renewal_R6_RenewalDateMissed_Eligible,
       CAST(NULL AS INT)
           AS Van_Renewal_R6_RenewalDateMissed_Affected,
       CAST(NULL AS INT)
           AS Van_Renewal_R6_RenewalDateMissed_Comparison,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_Eligible,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_Affected,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_Comparison,
       CAST(NULL AS STRING)
           AS Van_Change_C1_ChangeCallWait_Resolution,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_CallsAttempted,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_CallsConnected,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_CallsAbandoned,
       CAST(NULL AS INT)
           AS Van_Change_C1_ChangeCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(c3c.n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_DocRequest_C3_DocRequestCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Van_DocRequest_C3_DocRequestCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND (NOT (COALESCE(c3c.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Van_DocRequest_C3_DocRequestCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT ((COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN c3c.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Van_DocRequest_C3_DocRequestCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n END
           AS Van_DocRequest_C3_DocRequestCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n_conn END
           AS Van_DocRequest_C3_DocRequestCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n_aband END
           AS Van_DocRequest_C3_DocRequestCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.max_wait END
           AS Van_DocRequest_C3_DocRequestCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN COALESCE(c3c.n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_DocRequest_C4_DocNotArrived_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n, 0) > 1) THEN 1 ELSE 0 END
           AS Van_DocRequest_C4_DocNotArrived_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND (NOT (COALESCE(c3c.n, 0) > 1)) THEN 1 ELSE 0 END
           AS Van_DocRequest_C4_DocNotArrived_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) OR NOT ((COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n, 0) > 1)) THEN NULL ELSE 'Unknown' END
           AS Van_DocRequest_C4_DocNotArrived_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls}) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE COALESCE(c3c.n, 0) - 1 END
           AS Van_DocRequest_C4_DocNotArrived_EventCount,
       CAST(NULL AS INT)
           AS Van_DocRequest_C5_DocFromInsurer_Eligible,
       CAST(NULL AS INT)
           AS Van_DocRequest_C5_DocFromInsurer_Affected,
       CAST(NULL AS INT)
           AS Van_DocRequest_C5_DocFromInsurer_Comparison,
       CAST(NULL AS STRING)
           AS Van_DocRequest_C5_DocFromInsurer_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Loan}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.loan_reached, 0) = 1) OR COALESCE(ev.loan_n, 0) > 0) THEN 1 ELSE 0 END
           AS Van_FirstYear_VA5B1_LoanSetup_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Loan}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.loan_reached, 0) = 1) OR COALESCE(ev.loan_n, 0) > 0) AND COALESCE(ev.loan_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_FirstYear_VA5B1_LoanSetup_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Loan}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.loan_reached, 0) = 1) OR COALESCE(ev.loan_n, 0) > 0) AND NOT COALESCE(ev.loan_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_FirstYear_VA5B1_LoanSetup_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Loan}) OR e.is_open = 1) OR NOT ((COALESCE(ev.loan_reached, 0) = 1) OR COALESCE(ev.loan_n, 0) > 0) THEN NULL ELSE COALESCE(ev.loan_n, 0) END
           AS Van_FirstYear_VA5B1_LoanSetup_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) THEN 1 ELSE 0 END
           AS Van_FirstYear_VA6B1_DocIntakeAfterSale_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) AND COALESCE(ev.fy_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_FirstYear_VA6B1_DocIntakeAfterSale_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) AND NOT COALESCE(ev.fy_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_FirstYear_VA6B1_DocIntakeAfterSale_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_FyDoc}) OR e.is_open = 1) OR NOT ((COALESCE(ev.fy_doc_reached, 0) = 1) OR COALESCE(ev.fy_doc_n, 0) > 0) THEN NULL ELSE COALESCE(ev.fy_doc_n, 0) END
           AS Van_FirstYear_VA6B1_DocIntakeAfterSale_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) THEN 1 ELSE 0 END
           AS Van_Renewal_VR2B1_OnlineRequote_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) AND COALESCE(ev.requote_win_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Renewal_VR2B1_OnlineRequote_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) AND NOT COALESCE(ev.requote_win_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Renewal_VR2B1_OnlineRequote_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Requote}) OR e.is_open = 1) OR NOT ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.requote_win_n, 0) > 0) THEN NULL ELSE COALESCE(ev.requote_win_n, 0) END
           AS Van_Renewal_VR2B1_OnlineRequote_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) THEN NULL WHEN ((e.Outcome_ThisRenewal = 'Renewed') OR COALESCE(ev.rn_doc_n, 0) > 0) THEN 1 ELSE 0 END
           AS Van_Renewal_VR4B2_DocIntakeAfterRenewal_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) THEN NULL WHEN ((e.Outcome_ThisRenewal = 'Renewed') OR COALESCE(ev.rn_doc_n, 0) > 0) AND COALESCE(ev.rn_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Renewal_VR4B2_DocIntakeAfterRenewal_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) THEN NULL WHEN ((e.Outcome_ThisRenewal = 'Renewed') OR COALESCE(ev.rn_doc_n, 0) > 0) AND NOT COALESCE(ev.rn_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Renewal_VR4B2_DocIntakeAfterRenewal_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_RnDoc}) OR e.is_open = 1) OR NOT ((e.Outcome_ThisRenewal = 'Renewed') OR COALESCE(ev.rn_doc_n, 0) > 0) THEN NULL ELSE COALESCE(ev.rn_doc_n, 0) END
           AS Van_Renewal_VR4B2_DocIntakeAfterRenewal_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) THEN 1 ELSE 0 END
           AS Van_Change_VM3B1_DocIntakeForChange_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) AND COALESCE(ev.change_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Change_VM3B1_DocIntakeForChange_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) AND NOT COALESCE(ev.change_doc_n, 0) > 0 THEN 1 ELSE 0 END
           AS Van_Change_VM3B1_DocIntakeForChange_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_ChangeDoc}) OR e.is_open = 1) OR NOT ((COALESCE(ev.change_reached, 0) = 1) OR COALESCE(ev.change_doc_n, 0) > 0) THEN NULL ELSE COALESCE(ev.change_doc_n, 0) END
           AS Van_Change_VM3B1_DocIntakeForChange_EventCount
FROM   stg.fe3m_van_ep e
LEFT JOIN stg.fe3m_van_evagg ev ON ev.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_van_ev2   c2 ON c2.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_van_r3    r3 ON r3.EpisodeID = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_van_callagg WHERE m = 'F1') f1  ON f1.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_van_callagg WHERE m = 'R2') r2  ON r2.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_van_callagg WHERE m = 'C1') c1  ON c1.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_van_callagg WHERE m = 'C3') c3c ON c3c.EpisodeID = e.EpisodeID
;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Checks

Grain: one row per Van episode (expect duplicate_ids = 0).

In [10]:
SELECT (SELECT COUNT(*) FROM stg.fe3m_van) AS rows_, (SELECT COUNT(*) FROM stg.fe3m_van_ep) AS episodes,
       (SELECT COUNT(*) - COUNT(DISTINCT EpisodeID) FROM stg.fe3m_van) AS duplicate_ids;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

Per measure: rows filled, eligible, numerator, comparison, invariant violations (expect bad_groups = 0).

In [11]:
SELECT 'Van_FirstYear_F1_SalesCallWait' AS measure, COUNT(Van_FirstYear_F1_SalesCallWait_Eligible) AS rows_filled, SUM(Van_FirstYear_F1_SalesCallWait_Eligible) AS eligible, SUM(Van_FirstYear_F1_SalesCallWait_Affected) AS numerator,
       SUM(Van_FirstYear_F1_SalesCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_F1_SalesCallWait_Eligible = 1 AND COALESCE(Van_FirstYear_F1_SalesCallWait_Affected, 0) + COALESCE(Van_FirstYear_F1_SalesCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_FirstYear_F2_StartedOnlineFinishedByPhone' AS measure, COUNT(Van_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible) AS rows_filled, SUM(Van_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible) AS eligible, SUM(Van_FirstYear_F2_StartedOnlineFinishedByPhone_Affected) AS numerator,
       SUM(Van_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible = 1 AND COALESCE(Van_FirstYear_F2_StartedOnlineFinishedByPhone_Affected, 0) + COALESCE(Van_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_FirstYear_F3_NoPriceOnline' AS measure, COUNT(Van_FirstYear_F3_NoPriceOnline_Eligible) AS rows_filled, SUM(Van_FirstYear_F3_NoPriceOnline_Eligible) AS eligible, SUM(Van_FirstYear_F3_NoPriceOnline_Affected) AS numerator,
       SUM(Van_FirstYear_F3_NoPriceOnline_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_F3_NoPriceOnline_Eligible = 1 AND COALESCE(Van_FirstYear_F3_NoPriceOnline_Affected, 0) + COALESCE(Van_FirstYear_F3_NoPriceOnline_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_FirstYear_F4_DocsChasedByCustomer' AS measure, COUNT(Van_FirstYear_F4_DocsChasedByCustomer_Eligible) AS rows_filled, SUM(Van_FirstYear_F4_DocsChasedByCustomer_Eligible) AS eligible, SUM(Van_FirstYear_F4_DocsChasedByCustomer_Affected) AS numerator,
       SUM(Van_FirstYear_F4_DocsChasedByCustomer_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_F4_DocsChasedByCustomer_Eligible = 1 AND COALESCE(Van_FirstYear_F4_DocsChasedByCustomer_Affected, 0) + COALESCE(Van_FirstYear_F4_DocsChasedByCustomer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_FirstYear_F5_LoanSetupChase' AS measure, COUNT(Van_FirstYear_F5_LoanSetupChase_Eligible) AS rows_filled, SUM(Van_FirstYear_F5_LoanSetupChase_Eligible) AS eligible, SUM(Van_FirstYear_F5_LoanSetupChase_Affected) AS numerator,
       SUM(Van_FirstYear_F5_LoanSetupChase_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_F5_LoanSetupChase_Eligible = 1 AND COALESCE(Van_FirstYear_F5_LoanSetupChase_Affected, 0) + COALESCE(Van_FirstYear_F5_LoanSetupChase_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_R1_FailedLogin' AS measure, COUNT(Van_Renewal_R1_FailedLogin_Eligible) AS rows_filled, SUM(Van_Renewal_R1_FailedLogin_Eligible) AS eligible, SUM(Van_Renewal_R1_FailedLogin_Affected) AS numerator,
       SUM(Van_Renewal_R1_FailedLogin_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_R1_FailedLogin_Eligible = 1 AND COALESCE(Van_Renewal_R1_FailedLogin_Affected, 0) + COALESCE(Van_Renewal_R1_FailedLogin_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_R2_RenewalCallWait' AS measure, COUNT(Van_Renewal_R2_RenewalCallWait_Eligible) AS rows_filled, SUM(Van_Renewal_R2_RenewalCallWait_Eligible) AS eligible, SUM(Van_Renewal_R2_RenewalCallWait_Affected) AS numerator,
       SUM(Van_Renewal_R2_RenewalCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_R2_RenewalCallWait_Eligible = 1 AND COALESCE(Van_Renewal_R2_RenewalCallWait_Affected, 0) + COALESCE(Van_Renewal_R2_RenewalCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_R3_OnlineThenCalled' AS measure, COUNT(Van_Renewal_R3_OnlineThenCalled_Eligible) AS rows_filled, SUM(Van_Renewal_R3_OnlineThenCalled_Eligible) AS eligible, SUM(Van_Renewal_R3_OnlineThenCalled_Affected) AS numerator,
       SUM(Van_Renewal_R3_OnlineThenCalled_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_R3_OnlineThenCalled_Eligible = 1 AND COALESCE(Van_Renewal_R3_OnlineThenCalled_Affected, 0) + COALESCE(Van_Renewal_R3_OnlineThenCalled_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_R4_RequoteNoPrice' AS measure, COUNT(Van_Renewal_R4_RequoteNoPrice_Eligible) AS rows_filled, SUM(Van_Renewal_R4_RequoteNoPrice_Eligible) AS eligible, SUM(Van_Renewal_R4_RequoteNoPrice_Affected) AS numerator,
       SUM(Van_Renewal_R4_RequoteNoPrice_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_R4_RequoteNoPrice_Eligible = 1 AND COALESCE(Van_Renewal_R4_RequoteNoPrice_Affected, 0) + COALESCE(Van_Renewal_R4_RequoteNoPrice_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_R5_RenewalHeld' AS measure, COUNT(Van_Renewal_R5_RenewalHeld_Eligible) AS rows_filled, SUM(Van_Renewal_R5_RenewalHeld_Eligible) AS eligible, SUM(Van_Renewal_R5_RenewalHeld_Affected) AS numerator,
       SUM(Van_Renewal_R5_RenewalHeld_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_R5_RenewalHeld_Eligible = 1 AND COALESCE(Van_Renewal_R5_RenewalHeld_Affected, 0) + COALESCE(Van_Renewal_R5_RenewalHeld_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_R6_RenewalDateMissed' AS measure, COUNT(Van_Renewal_R6_RenewalDateMissed_Eligible) AS rows_filled, SUM(Van_Renewal_R6_RenewalDateMissed_Eligible) AS eligible, SUM(Van_Renewal_R6_RenewalDateMissed_Affected) AS numerator,
       SUM(Van_Renewal_R6_RenewalDateMissed_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_R6_RenewalDateMissed_Eligible = 1 AND COALESCE(Van_Renewal_R6_RenewalDateMissed_Affected, 0) + COALESCE(Van_Renewal_R6_RenewalDateMissed_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Change_C1_ChangeCallWait' AS measure, COUNT(Van_Change_C1_ChangeCallWait_Eligible) AS rows_filled, SUM(Van_Change_C1_ChangeCallWait_Eligible) AS eligible, SUM(Van_Change_C1_ChangeCallWait_Affected) AS numerator,
       SUM(Van_Change_C1_ChangeCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Van_Change_C1_ChangeCallWait_Eligible = 1 AND COALESCE(Van_Change_C1_ChangeCallWait_Affected, 0) + COALESCE(Van_Change_C1_ChangeCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_DocRequest_C3_DocRequestCallWait' AS measure, COUNT(Van_DocRequest_C3_DocRequestCallWait_Eligible) AS rows_filled, SUM(Van_DocRequest_C3_DocRequestCallWait_Eligible) AS eligible, SUM(Van_DocRequest_C3_DocRequestCallWait_Affected) AS numerator,
       SUM(Van_DocRequest_C3_DocRequestCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Van_DocRequest_C3_DocRequestCallWait_Eligible = 1 AND COALESCE(Van_DocRequest_C3_DocRequestCallWait_Affected, 0) + COALESCE(Van_DocRequest_C3_DocRequestCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_DocRequest_C4_DocNotArrived' AS measure, COUNT(Van_DocRequest_C4_DocNotArrived_Eligible) AS rows_filled, SUM(Van_DocRequest_C4_DocNotArrived_Eligible) AS eligible, SUM(Van_DocRequest_C4_DocNotArrived_Affected) AS numerator,
       SUM(Van_DocRequest_C4_DocNotArrived_Comparison) AS comparison,
       SUM(CASE WHEN Van_DocRequest_C4_DocNotArrived_Eligible = 1 AND COALESCE(Van_DocRequest_C4_DocNotArrived_Affected, 0) + COALESCE(Van_DocRequest_C4_DocNotArrived_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_DocRequest_C5_DocFromInsurer' AS measure, COUNT(Van_DocRequest_C5_DocFromInsurer_Eligible) AS rows_filled, SUM(Van_DocRequest_C5_DocFromInsurer_Eligible) AS eligible, SUM(Van_DocRequest_C5_DocFromInsurer_Affected) AS numerator,
       SUM(Van_DocRequest_C5_DocFromInsurer_Comparison) AS comparison,
       SUM(CASE WHEN Van_DocRequest_C5_DocFromInsurer_Eligible = 1 AND COALESCE(Van_DocRequest_C5_DocFromInsurer_Affected, 0) + COALESCE(Van_DocRequest_C5_DocFromInsurer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_FirstYear_VA5B1_LoanSetup' AS measure, COUNT(Van_FirstYear_VA5B1_LoanSetup_Eligible) AS rows_filled, SUM(Van_FirstYear_VA5B1_LoanSetup_Eligible) AS eligible, SUM(Van_FirstYear_VA5B1_LoanSetup_Entered) AS numerator,
       SUM(Van_FirstYear_VA5B1_LoanSetup_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_VA5B1_LoanSetup_Eligible = 1 AND COALESCE(Van_FirstYear_VA5B1_LoanSetup_Entered, 0) + COALESCE(Van_FirstYear_VA5B1_LoanSetup_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_FirstYear_VA6B1_DocIntakeAfterSale' AS measure, COUNT(Van_FirstYear_VA6B1_DocIntakeAfterSale_Eligible) AS rows_filled, SUM(Van_FirstYear_VA6B1_DocIntakeAfterSale_Eligible) AS eligible, SUM(Van_FirstYear_VA6B1_DocIntakeAfterSale_Entered) AS numerator,
       SUM(Van_FirstYear_VA6B1_DocIntakeAfterSale_Comparison) AS comparison,
       SUM(CASE WHEN Van_FirstYear_VA6B1_DocIntakeAfterSale_Eligible = 1 AND COALESCE(Van_FirstYear_VA6B1_DocIntakeAfterSale_Entered, 0) + COALESCE(Van_FirstYear_VA6B1_DocIntakeAfterSale_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_VR2B1_OnlineRequote' AS measure, COUNT(Van_Renewal_VR2B1_OnlineRequote_Eligible) AS rows_filled, SUM(Van_Renewal_VR2B1_OnlineRequote_Eligible) AS eligible, SUM(Van_Renewal_VR2B1_OnlineRequote_Entered) AS numerator,
       SUM(Van_Renewal_VR2B1_OnlineRequote_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_VR2B1_OnlineRequote_Eligible = 1 AND COALESCE(Van_Renewal_VR2B1_OnlineRequote_Entered, 0) + COALESCE(Van_Renewal_VR2B1_OnlineRequote_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Renewal_VR4B2_DocIntakeAfterRenewal' AS measure, COUNT(Van_Renewal_VR4B2_DocIntakeAfterRenewal_Eligible) AS rows_filled, SUM(Van_Renewal_VR4B2_DocIntakeAfterRenewal_Eligible) AS eligible, SUM(Van_Renewal_VR4B2_DocIntakeAfterRenewal_Entered) AS numerator,
       SUM(Van_Renewal_VR4B2_DocIntakeAfterRenewal_Comparison) AS comparison,
       SUM(CASE WHEN Van_Renewal_VR4B2_DocIntakeAfterRenewal_Eligible = 1 AND COALESCE(Van_Renewal_VR4B2_DocIntakeAfterRenewal_Entered, 0) + COALESCE(Van_Renewal_VR4B2_DocIntakeAfterRenewal_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van
UNION ALL
SELECT 'Van_Change_VM3B1_DocIntakeForChange' AS measure, COUNT(Van_Change_VM3B1_DocIntakeForChange_Eligible) AS rows_filled, SUM(Van_Change_VM3B1_DocIntakeForChange_Eligible) AS eligible, SUM(Van_Change_VM3B1_DocIntakeForChange_Entered) AS numerator,
       SUM(Van_Change_VM3B1_DocIntakeForChange_Comparison) AS comparison,
       SUM(CASE WHEN Van_Change_VM3B1_DocIntakeForChange_Eligible = 1 AND COALESCE(Van_Change_VM3B1_DocIntakeForChange_Entered, 0) + COALESCE(Van_Change_VM3B1_DocIntakeForChange_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_van;

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 21, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 6 fields>

In [12]:
%%pyspark
mssparkutils.session.stop()

StatementMeta(, b396ac87-0f59-40b7-9889-752c1d819590, 23, Finished, Available, Finished, False)